# Drug Side Effects - Pipeline de 3 Agentes con LangChain + Mistral AI

Pipeline completo de 3 agentes especializados:

| Agente | Funcion |
|--------|--------|
| Agente 1 - Normalizador | Limpia, imputa, escala y codifica el dataset |
| Agente 2 - Entrenador | Entrena modelos con validacion, selecciona metricas y el mejor modelo |
| Agente 3 - Comunicador | Genera un reporte en lenguaje natural con los resultados |

Modelo: mistral-small-latest via LangChain  
Dataset: Drug Side Effects (100k registros)

## PASO 1 - Instalacion de dependencias

In [ ]:
%%capture
!pip install numpy==1.26.4 scikit-learn==1.5.2 --force-reinstall --quiet 2>/dev/null
!pip install langchain langchain-mistralai langchain-experimental --quiet
!pip install pandas openpyxl tabulate --upgrade --quiet

print('Instalacion completada')

## PASO 2 - Configurar API Key de Mistral AI

In [ ]:
import os
from getpass import getpass

MIAPIKEI = getpass('Ingresa tu Mistral API Key: ')
os.environ['MIAPIKEI'] = MIAPIKEI

print(f'API Key configurada ({len(MIAPIKEI)} chars)')

## PASO 3 - Inicializar el modelo Mistral

In [ ]:
from langchain_mistralai import ChatMistralAI
import time
import httpx
import warnings
warnings.filterwarnings('ignore')

llm = ChatMistralAI(
    model='mistral-small-latest',
    temperature=0,
    api_key=MIAPIKEI
)

for intento in range(3):
    try:
        resp = llm.invoke('Decime Listo para procesar datos en espanol.')
        print(f'Mistral: {resp.content}')
        break
    except httpx.HTTPStatusError as e:
        if e.response.status_code == 429:
            espera = 10 * (2 ** intento)
            print(f'Rate limit. Esperando {espera}s...')
            time.sleep(espera)
        else:
            raise

## PASO 4 - Cargar el Dataset de Efectos Secundarios

In [ ]:
import pandas as pd
import numpy as np

df_raw = pd.read_csv('drug_side_effects_100k_dataset.csv')

print(f'Dataset: {df_raw.shape[0]} filas x {df_raw.shape[1]} columnas')
print(f'Columnas: {list(df_raw.columns)}')
df_raw.head()

## PASO 5 - Exploracion rapida del Dataset

In [ ]:
print('='*60)
print('INFO DEL DATASET')
print('='*60)

info_df = pd.DataFrame({
    'Dtype':       df_raw.dtypes,
    'Nulos':       df_raw.isnull().sum(),
    '% Nulos':     (df_raw.isnull().mean() * 100).round(2),
    'Unicos':      df_raw.nunique()
})
display(info_df)

print()
print('='*60)
print('ESTADISTICAS NUMERICAS')
print('='*60)
display(df_raw.describe(include='all').T)

## Funcion helper - llamada segura con reintentos

In [ ]:
def invoke_with_retry(agent, prompt, max_retries=6, base_wait=10):
    for intento in range(max_retries):
        try:
            return agent.invoke({'input': prompt})
        except httpx.HTTPStatusError as e:
            if e.response.status_code == 429:
                espera = base_wait * (2 ** intento)
                print(f'Rate limit ({intento+1}/{max_retries}). Esperando {espera}s...')
                time.sleep(espera)
            else:
                raise
    raise Exception('Se agotaron los reintentos por rate limit')

print('Helper listo')

---
## AGENTE 1 - Normalizador

Objetivo: Limpiar, imputar valores faltantes, escalar numericas y codificar categoricas.

Usa create_pandas_dataframe_agent con instrucciones para evitar errores de pandas.

In [ ]:
from langchain_experimental.agents import create_pandas_dataframe_agent

agent_normalizador = create_pandas_dataframe_agent(
    llm                  = llm,
    df                   = df_raw,
    agent_type           = 'tool-calling',
    verbose              = True,
    allow_dangerous_code = True,
    prefix               = '''
Eres un experto en limpieza y preprocesamiento de datos con pandas.

REGLAS CRITICAS (NO USAR INPLACE):
- NUNCA uses inplace=True. Siempre asignacion directa.
- MAL: df['col'].fillna(valor, inplace=True)
- BIEN: df['col'] = df['col'].fillna(valor)
- MAL: df.drop(columns=['x'], inplace=True)
- BIEN: df = df.drop(columns=['x'])

Trabajas con un DataFrame 'df' de efectos secundarios de medicamentos.
Siempre hace una copia al inicio: df_limpio = df.copy()
Al final de cada paso, muestra un resumen de lo que cambio.
Al finalizar todo, la variable df_limpio debe estar disponible.
'''
)

print('Agente Normalizador creado')

In [ ]:
def ejecutar_normalizacion():
    pasos = [
        '''
        Hace df_limpio = df.copy().
        Muestra cuantas filas y columnas tiene.
        ''',
        '''
        Analiza los valores nulos en df_limpio.
        Para numericas con nulos: df_limpio['col'] = df_limpio['col'].fillna(mediana).
        Para categoricas con nulos: df_limpio['col'] = df_limpio['col'].fillna(moda).
        Muestra nulos antes y despues.
        ''',
        '''
        Identifica las columnas categoricas (tipo 'object') en df_limpio.
        Excluye 'patient_id', 'report_date', 'treatment_start_date'.
        Aplica pd.get_dummies() a las categoricas con menos de 10 valores unicos.
        Al resto, aplica Label Encoding con df['col'] = df['col'].astype('category').cat.codes.
        Muestra cuantas columnas habia antes y despues.
        ''',
        '''
        Identifica las columnas numericas en df_limpio.
        Excluye 'patient_id' si existe.
        Aplica StandardScaler:
        from sklearn.preprocessing import StandardScaler
        scaler = StandardScaler()
        cols_num = [lista de columnas numericas]
        df_limpio[cols_num] = scaler.fit_transform(df_limpio[cols_num])
        Muestra medias y desvios antes/despues.
        ''',
        '''
        Informa el estado final de df_limpio:
        - Dimensiones
        - Columnas numericas vs categoricas
        - Verifica que no haya nulos
        - Muestra las primeras 3 filas
        '''
    ]

    for i, paso in enumerate(pasos, 1):
        print()
        print('='*60)
        print(f'Paso {i}/{len(pasos)}')
        print('='*60)
        invoke_with_retry(agent_normalizador, paso)
        print(f'Paso {i} completado')

ejecutar_normalizacion()

In [ ]:
try:
    print(f'df_limpio: {df_limpio.shape}, nulos: {df_limpio.isnull().sum().sum()}')
    df_limpio.head()
except NameError:
    print('df_limpio no definido. Ejecuta la celda anterior.')

---
## AGENTE 2 - Entrenador

Objetivo: Entrenar modelos de clasificacion, validar con cross-validation, seleccionar metricas y elegir el mejor modelo.

Usa LangChain agent + scikit-learn. Target: severity (Mild, Moderate, Severe).

In [ ]:
try:
    df_ml = df_limpio.copy()
except NameError:
    df_ml = df_raw.copy()
    print('Usando df_raw (df_limpio no disponible)')

target_col = 'severity'

if target_col in df_ml.columns:
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    df_ml[target_col] = le.fit_transform(df_ml[target_col].astype(str))
    print(f'Target codificado: {dict(enumerate(le.classes_))}')

    exclude_cols = [target_col, 'patient_id', 'report_date', 'treatment_start_date']
    feature_cols = [c for c in df_ml.columns if c not in exclude_cols]

    for col in df_ml[feature_cols].select_dtypes(include=['object', 'category']).columns:
        df_ml[col] = LabelEncoder().fit_transform(df_ml[col].astype(str))

    print(f'Features: {len(feature_cols)} columnas numericas')
    print(f'Target: {df_ml[target_col].nunique()} clases')
else:
    print(f'Columna {target_col} no encontrada')

### Crear Agente Entrenador

El agente ejecuta codigo Python via sklearn y guarda los resultados en variables globales que persisten fuera del agente.

In [ ]:
agent_entrenador = create_pandas_dataframe_agent(
    llm                  = llm,
    df                   = df_ml,
    agent_type           = 'tool-calling',
    verbose              = True,
    allow_dangerous_code = True,
    prefix               = '''
Eres un experto en Machine Learning. La variable target es 'severity'.

IMPORTANTE: usa from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
usa from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
usa from sklearn.linear_model import LogisticRegression
usa from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
usa from sklearn.preprocessing import LabelEncoder

SIEMPRE usa train_test_split con stratify, 5-fold cross-validation.

ALMACENA resultados en 'resultados_modelos' como lista de diccionarios.
CADA diccionario debe tener: 'modelo', 'cv_accuracy', 'cv_f1', 'test_accuracy', 'test_precision', 'test_recall', 'test_f1'

Elige el mejor modelo por f1-score macro y guardalo como string en 'mejor_modelo'.

REGLAS:
- NO uses inplace=True
- Si un modelo da error, prueba el siguiente
- Muestra matriz de confusion del mejor modelo
CUANDO termines todos los pasos, imprime 'ENTRENAMIENTO FINALIZADO'
'''
)

print('Agente Entrenador creado')

In [ ]:
def ejecutar_entrenamiento():
    pasos = [
        '''
        Divide los datos:
        X = todas las columnas excepto severity, patient_id, report_date, treatment_start_date
        y = columna severity
        train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
        Muestra cuantas muestras en train y test.
        ''',
        '''
        Entrena LogisticRegression(max_iter=1000).
        Evaluacion cruzada: cross_val_score(modelo, X_train, y_train, cv=5, scoring='f1_macro')
        Evaluacion en test: accuracy, precision, recall, f1.
        Guarda en resultados_modelos.
        ''',
        '''
        Entrena RandomForestClassifier(n_estimators=100, random_state=42).
        Evaluacion cruzada y en test.
        Guarda en resultados_modelos.
        ''',
        '''
        Entrena GradientBoostingClassifier(n_estimators=100, random_state=42).
        Evaluacion cruzada y en test.
        Guarda en resultados_modelos.
        ''',
        '''
        Compara todos los modelos en resultados_modelos.
        Muestra una tabla con accuracy, precision, recall, f1 de cada uno.
        Elige el mejor por f1-score macro.
        Guarda su nombre como string en 'mejor_modelo'.
        Muestra la matriz de confusion del mejor modelo.
        Imprime ENTRENAMIENTO FINALIZADO al terminar.
        '''
    ]

    for i, paso in enumerate(pasos, 1):
        print()
        print('='*60)
        print(f'Paso {i}/{len(pasos)}')
        print('='*60)
        invoke_with_retry(agent_entrenador, paso)
        print(f'Paso {i} completado')

ejecutar_entrenamiento()

In [ ]:
try:
    print('='*60)
    print('RESULTADOS DE MODELOS')
    print('='*60)
    display(pd.DataFrame(resultados_modelos))
except NameError:
    print('resultados_modelos no encontrado')

try:
    print(f'Mejor modelo: {mejor_modelo}')
except NameError:
    print('mejor_modelo no definido')

---
## AGENTE 3 - Comunicador

Objetivo: Generar un reporte en lenguaje natural con todos los resultados del pipeline.

Usa Mistral + LangChain con un prompt estructurado.

In [ ]:
def construir_contexto_reporte():
    ctx = []

    ctx.append('=== DATASET ORIGINAL ===')
    ctx.append(f'Filas: {df_raw.shape[0]}, Columnas: {df_raw.shape[1]}')
    ctx.append(f'Columnas: {list(df_raw.columns)}')
    ctx.append(f'Nulos originales: {df_raw.isnull().sum().sum()}')

    try:
        ctx.append('')
        ctx.append('=== DATASET NORMALIZADO ===')
        ctx.append(f'Filas: {df_limpio.shape[0]}, Columnas: {df_limpio.shape[1]}')
        ctx.append(f'Nulos despues: {df_limpio.isnull().sum().sum()}')
    except NameError:
        ctx.append('')
        ctx.append('=== DATASET NORMALIZADO: No disponible ===')

    try:
        ctx.append('')
        ctx.append('=== RESULTADOS DE MODELOS ===')
        for m in resultados_modelos:
            ctx.append(f"- {m.get('modelo', '?'):20} | "
                f"Acc={m.get('test_accuracy', 0):.4f} | "
                f"Prec={m.get('test_precision', 0):.4f} | "
                f"Rec={m.get('test_recall', 0):.4f} | "
                f"F1={m.get('test_f1', 0):.4f}")
    except NameError:
        ctx.append('')
        ctx.append('=== RESULTADOS DE MODELOS: No disponibles ===')

    try:
        ctx.append('')
        ctx.append(f'=== MEJOR MODELO ===')
        ctx.append(f'{mejor_modelo}')
    except NameError:
        ctx.append('')
        ctx.append('=== MEJOR MODELO: No seleccionado ===')

    return '\n'.join(ctx)

contexto = construir_contexto_reporte()
print(contexto)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt_reporte = ChatPromptTemplate.from_messages([
    ('system', '''
Sos un Analista de Datos Senior especializado en farmacovigilancia.
Genera un reporte profesional en markdown con:
1. Resumen Ejecutivo
2. Preprocesamiento: nulos, codificacion, escalado
3. Modelos entrenados y sus metricas (tabla)
4. Modelo seleccionado y justificacion
5. Recomendaciones
Usa numeros concretos, sin placeholders.
Usa el formato markdown de manera profesional.
'''),
    ('user', '{contexto}')
])

for intento in range(3):
    try:
        respuesta = llm.invoke(prompt_reporte.format_messages(contexto=contexto))
        reporte_final = respuesta.content
        break
    except httpx.HTTPStatusError as e:
        if e.response.status_code == 429:
            espera = 15 * (2 ** intento)
            print(f'Rate limit. Esperando {espera}s...')
            time.sleep(espera)
        else:
            raise

print('='*60)
print('REPORTE GENERADO')
print('='*60)
print(reporte_final)

In [ ]:
with open('reporte_efectos_secundarios.md', 'w', encoding='utf-8') as f:
    f.write(reporte_final)

print(f'Reporte guardado: reporte_efectos_secundarios.md')
print(f'Total: {len(reporte_final)} caracteres')

---
## Pipeline completo

Ejecuta todo en secuencia.

In [ ]:
print('INICIANDO PIPELINE COMPLETO')

# --- AGENTE 1 ---
print('='*60)
print('AGENTE 1 - NORMALIZADOR')
print('='*60)
ejecutar_normalizacion()

# --- AGENTE 2 ---
print()
print('='*60)
print('AGENTE 2 - ENTRENADOR')
print('='*60)

df_ml = df_limpio.copy()
le = LabelEncoder()
df_ml[target_col] = le.fit_transform(df_ml[target_col].astype(str))
exclude_cols = [target_col, 'patient_id', 'report_date', 'treatment_start_date']
feature_cols = [c for c in df_ml.columns if c not in exclude_cols]
for col in df_ml[feature_cols].select_dtypes(include=['object', 'category']).columns:
    df_ml[col] = LabelEncoder().fit_transform(df_ml[col].astype(str))

agent_entrenador = create_pandas_dataframe_agent(
    llm=llm, df=df_ml, agent_type='tool-calling',
    verbose=True, allow_dangerous_code=True,
    prefix='Eres un experto en ML. Target: severity. No uses inplace=True.'
)
ejecutar_entrenamiento()

# --- AGENTE 3 ---
print()
print('='*60)
print('AGENTE 3 - COMUNICADOR')
print('='*60)

contexto = construir_contexto_reporte()
for intento in range(3):
    try:
        respuesta = llm.invoke(prompt_reporte.format_messages(contexto=contexto))
        reporte_final = respuesta.content
        break
    except httpx.HTTPStatusError as e:
        if e.response.status_code == 429:
            time.sleep(15 * (2 ** intento))
        else:
            raise

with open('reporte_efectos_secundarios.md', 'w', encoding='utf-8') as f:
    f.write(reporte_final)

print()
print('='*60)
print('PIPELINE COMPLETADO')
print('='*60)
print()
print('reporte_efectos_secundarios.md generado')
print(reporte_final)